In [0]:
%sql
--#load the data into the storage
CREATE CATALOG Logistics_catalog_assign;
CREATE SCHEMA IF NOT EXISTS logistics_catalog_assign.landing_zone;
CREATE VOLUME IF NOT EXISTS logistics_catalog_assign.landing_zone.landing_vol;


In [0]:
#Finding an couple of data patterns applying below EDA
df1 = spark.read.csv("/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source1.txt",sep=',',inferSchema=True,header=False).toDF("Shipment_id","FirstName","LastName","Age","Role")
df1.display()
df1.summary()

In [0]:
from pyspark.sql import SparkSession

def get_spark_session(app_name="Logistics Data Engineering Project"):
    """
    Creates and returns a SparkSession.
    Works in Databricks and non-Databricks environments.
    """
    try:
        spark = SparkSession.getActiveSession()
        if spark:
            return spark
    except:
        pass

    return (SparkSession.builder.appName(app_name).getOrCreate())

def read_csv_df(spark,path,header=True,infer_schema=True,sep=","):
    return (spark.read
        .option("header", header)
        .option("inferSchema", infer_schema)
        .option("sep", sep)
        .csv(path))

spark=get_spark_session("Logistics Data Engineering Project")
df1=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source1.txt',True,True,',')
df1.display()
df2=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source2.txt',True,True,',')
df2.display()

In [0]:
df1=df1.dropDuplicates()
df2=df2.dropDuplicates()
source1 = df1.unionByName(df2, allowMissingColumns=True)
source1.display()

In [0]:
#Combining Data + Schema Merging (Structuring) - Active Data Munging

from pyspark.sql.functions import lit
def add_literal_columns(df, columns, default_value=None):
    for col_name in columns:
        df = df.withColumn(col_name, lit(default_value))
    return df

def read_csv_df(spark,path,header=True,infer_schema=True,sep=","):
    return (spark.read
        .option("header", header)
        .option("inferSchema", infer_schema)
        .option("sep", sep)
        .csv(path))

df1=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source1.txt',True,True,',')
df2=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source2.txt',True,True,',')

sourcedf1=add_literal_columns(df1,['source_system'],default_value='system1')
sourcedf2=add_literal_columns(df2,['source_system'],default_value='system2')
source1 = sourcedf1.unionByName(sourcedf2, allowMissingColumns=True)
source1.display()


In [0]:
#Cleansing and scrubbing - Active Data Munging
#Cleansing (removal of unwanted datasets)

def clean_srubbing(df, droprules):
    for subset, how in droprules:
        df = df.na.drop(how=how, subset=subset)
    return df


In [0]:
droprules=[(['first_name','last_name'],'all'),(['shipment_id','role'],'any'),(['shipment_id'],'all')]
source1=clean_srubbing(source1,droprules)
source1.display()


In [0]:
#Scrubbing (convert raw to tidy)
def scrubbing(df,fill_rules,rpl_rules):
    for sourcemap, subset in fill_rules:
        df = df.na.fill(sourcemap,subset=subset)
    for sourcemap1, subset in rpl_rules:
        df = df.na.replace(sourcemap1,subset=subset)
    return df

In [0]:
age_fill = {"age":"-1"}
vechicle_fill = {"vehicle_type":"UNKNOWN"}
age_rpl = {"ten":"-1", "":"-1"}
vechicle_rpl = {"Turck":"LMV","Bike":"TwoWheeler"}
fill_rules = [(age_fill, ["age"]), (vechicle_fill, ["vehicle_type"])]
rpl_rules = [(age_rpl, ["age"]), (vechicle_rpl, ["vehicle_type"])]
source_cleaned = scrubbing(source1, fill_rules, rpl_rules)
source_cleaned.display()

In [0]:
#Standardization, De-Duplication and Replacement / Deletion of Data to make it in a usable format

#Standardizations:
def read_json(apark,path,mline):
    df = spark.read.json(path, multiLine=mline)
    return df

jsondf=read_json(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_shipment_detail_3000.json',True)
jsondf.display()


In [0]:
#Standardizations:

def addcol(df,colvalues):
    for col,val in colvalues:
       df=df.withColumn(col,lit(val))
    return df

def case_convrt(df,colname,casetype):
    if(casetype == "upper"):
        df=df.withColumn(colname,upper(col(colname)))
    elif(casetype == "lower"):
        df=df.withColumn(colname,lower(col(colname)))
    elif(casetype == "Intcap"):
        df=df.withColumn(colname,initcap(col(colname)))
    return df

from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import lit
from pyspark.sql.functions import upper,lower,initcap

colvalues=[("domain","Logistics"),("current timestamp",current_timestamp()),("is_expedited","False")]
df_stand=addcol(jsondf,colvalues)
df_stand=case_convrt(df_stand,"vehicle_type","upper")
source_clean_stand = case_convrt(source_cleaned,"role","lower")
source_clean_stand = case_convrt(source_cleaned,"hub_location","Intcap")
df_stand.display()
source_clean_stand.display()

In [0]:
#Format Standardization:
from pyspark.sql.functions import try_to_date,col,format_number
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
def formatting(df,format_rules):
    for formatrule, subset in format_rules:
        df = df.withColumn(subset, formatrule)
    return df
format_rules = [(try_to_date(col("shipment_date"), "yy-MM-dd"), "shipment_date")]
round_rules = [(format_number(col("shipment_cost"), 2), "shipment_cost")]
df_stand = formatting(df_stand, format_rules)
df_stand = formatting(df_stand, round_rules)
df_stand.display()



In [0]:
#Data Type Standardization
 
def datatype_change(source,colnames,typenames):
    for colname,typename in zip(colnames,typenames):
        source = source.withColumn(colname,source[colname].cast(typename))
    return source
 
colnames=['shipment_weight_kg','is_expedited']
typenames=['double','boolean']
df_stand=datatype_change(df_stand,colnames,typenames)
df_stand.display()
colnames=['age']
typenames=['int']
source_clean_stand=datatype_change(source_clean_stand,colnames,typenames)
source_clean_stand.display()


In [0]:
#Naming Standardization

def rename(df,rename_rules):
    for old,new in rename_rules:
        df=df.withColumnRenamed(old,new)
    return df

rename_rules=[('first_name','staff_first_name'),('last_name','staff_last_name'),('hub_location','origin_hub_city')]
source_clean_stand=rename(source_clean_stand,rename_rules)
source_clean_stand.display()


In [0]:
#Reordering columns logically in a better standard format:
final_df = df_stand.join(source_clean_stand,how='inner',on='shipment_id')
final_df = final_df.select('shipment_id','staff_first_name','staff_last_name','role','origin_hub_city','shipment_cost','current timestamp')
final_df.display()

In [0]:
#Deduplication
#Record Level Duplicates
final_df=final_df.dropDuplicates()
#Column Level Duplicates
final_df=final_df.dropDuplicates(["shipment_id"])
final_df.display()